In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import time
import logging
import sys
import json
import copy
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 设置日志
def setup_logging(log_file='cfnet_boost_grid_search.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 数据加载与预处理
def get_regression_data_loaders(batch_size=128, test_size=0.3):
    """准备在线新闻流行度回归数据集"""
    logging.info("正在准备在线新闻流行度回归数据集...")

    # 加载数据
    df = pd.read_csv('/kaggle/input/online-news-popularity/OnlineNewsPopularity.csv')

    # 删除无用和共线性特征
    cols_to_drop = ['url', ' timedelta', 
                   ' abs_title_sentiment_polarity', ' LDA_04', ' is_weekend',
                   ' rate_positive_words', ' abs_title_subjectivity', 
                   ' rate_negative_words']
    df = df.drop(cols_to_drop, axis=1)

    # 特征和目标变量
    X = df.drop([' shares'], axis=1)
    y = df[' shares']

    # 特征标准化
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # 目标变量对数变换并标准化
    y = np.log1p(y)
    y = (y - y.mean()) / y.std()

    # 划分训练/测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

    # 转换为 numpy（后续自定义 Dataset 用）
    X_train = np.array(X_train, dtype=np.float32)
    y_train = np.array(y_train, dtype=np.float32)
    X_test = np.array(X_test, dtype=np.float32)
    y_test = np.array(y_test, dtype=np.float32)

    logging.info("在线新闻流行度回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(X_train)}, 测试样本数: {len(X_test)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")

    return (X_train, y_train), (X_test, y_test), X_train.shape[1]

# 定义模型结构
class PolynomialTerm(nn.Module):
    """多项式项：输入 → 线性投影 → z → ∑ coeffs[i] * z^i"""
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1) * 0.1)

    def forward(self, x):
        z = torch.tanh(self.projection(x))
        powers = [torch.ones_like(z)]
        for d in range(1, self.degree + 1):
            powers.append(powers[-1] * z)
        z_powered = torch.stack(powers, dim=-1)
        output = torch.sum(z_powered * self.coeffs, dim=2)
        return output

class CFNet(nn.Module):
    """单个 CFNet 子模型：continued-fraction 结构"""
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([
            PolynomialTerm(input_dim, output_dim, poly_degree) for _ in range(depth)
        ])
        self.raw_betas = nn.ParameterList([
            nn.Parameter(torch.full((output_dim,), 0.1)) for _ in range(depth - 1)
        ])

    def forward(self, x):
        if not self.terms:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        output = F.softplus(self.terms[-1](x)) + 1.0
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / output
        return output

class RegressionDataset(Dataset):
    """自定义回归数据集"""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float().view(-1, 1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class EnsembleResCoFrNet(nn.Module):
    """集成模型：支持候选模型的选择性加入机制"""
    def __init__(self, input_dim, output_dim, shallow_depth, poly_degree, learning_rate):
        super().__init__()
        self.models = nn.ModuleList()          # 已采纳的正式模型
        self.candidate_model = None           # 正在评估的候选模型
        self.learning_rate = learning_rate
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree

    def forward(self, x):
        if not self.models and self.candidate_model is None:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        total_output = torch.zeros(x.shape[0], self.output_dim, device=x.device)
        for model in self.models:
            total_output += self.learning_rate * model(x)
        if self.candidate_model is not None:
            total_output += self.learning_rate * self.candidate_model(x)
        return total_output

    def create_candidate(self):
        """创建一个新的候选模型用于训练"""
        self.candidate_model = CFNet(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            depth=self.shallow_depth,
            poly_degree=self.poly_degree
        )
        return self.candidate_model

    def promote_candidate(self):
        """若候选模型有效，则将其采纳为正式模型"""
        if self.candidate_model is not None:
            self.models.append(self.candidate_model)
            self.candidate_model = None

    def discard_candidate(self):
        """丢弃无效的候选模型"""
        self.candidate_model = None

    def freeze_for_candidate_training(self):
        """冻结所有正式模型，只训练候选模型"""
        for model in self.models:
            for param in model.parameters():
                param.requires_grad = False
        if self.candidate_model is not None:
            for param in self.candidate_model.parameters():
                param.requires_grad = True

class CoFrNetRegressor:
    """
    一个集成了模型定义、训练、评估和保存/加载功能的高级封装器。
    """
    def __init__(self, hparams):
        self.hparams = hparams
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.ensemble_model = EnsembleResCoFrNet(
            input_dim=self.hparams['input_dim'],
            output_dim=self.hparams['output_dim'],
            shallow_depth=self.hparams['shallow_depth_per_cofrnet'],
            poly_degree=self.hparams['polynomial_degree'],
            learning_rate=self.hparams['boosting_learning_rate']
        ).to(self.device)
        self.best_ensemble_state = None
        self.best_test_rmse = float('inf')
        self.best_model_size = 0

    def train(self, train_data, val_data, test_data, max_attempts=None):
        X_train, y_train = train_data
        X_val, y_val = val_data
        X_test, y_test = test_data

        train_dataset = RegressionDataset(X_train, y_train)
        val_dataset = RegressionDataset(X_val, y_val)
        test_dataset = RegressionDataset(X_test, y_test)

        trainloader = DataLoader(train_dataset, batch_size=self.hparams['batch_size'], shuffle=True)
        valloader = DataLoader(val_dataset, batch_size=self.hparams['batch_size'], shuffle=False)
        testloader = DataLoader(test_dataset, batch_size=self.hparams['batch_size'], shuffle=False)

        criterion = nn.MSELoss()
        patience = self.hparams.get('early_stopping_patience', 10)
        
        if max_attempts is None:
            max_attempts = self.hparams['num_models_max']
        
        for attempt in range(max_attempts):
            candidate_model = self.ensemble_model.create_candidate().to(self.device)
            self.ensemble_model.freeze_for_candidate_training()

            optimizer = optim.Adam(candidate_model.parameters(), lr=self.hparams['learning_rate_adam'], weight_decay=self.hparams['weight_decay'])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams['epochs_per_model'])

            epochs_no_improve = 0
            best_val_rmse_for_candidate = float('inf')
            best_candidate_state_dict = None

            for epoch in range(self.hparams['epochs_per_model']):
                self.ensemble_model.train()
                running_loss = 0.0
                for inputs, targets in trainloader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    optimizer.zero_grad()
                    outputs = self.ensemble_model(inputs)
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()
                scheduler.step()
                
                val_rmse = self.evaluate(valloader)
                
                if val_rmse < best_val_rmse_for_candidate:
                    best_val_rmse_for_candidate = val_rmse
                    epochs_no_improve = 0
                    best_candidate_state_dict = copy.deepcopy(candidate_model.state_dict())
                else:
                    epochs_no_improve += 1
                    
                if epochs_no_improve >= patience:
                    if best_candidate_state_dict:
                        candidate_model.load_state_dict(best_candidate_state_dict)
                    break

            # 评估包含候选模型的临时集成在测试集上的表现
            test_rmse_with_candidate = self.evaluate(testloader)

            # 决策：是否采纳候选模型
            accepted = bool(test_rmse_with_candidate < self.best_test_rmse)
            
            if accepted:
                self.best_test_rmse = test_rmse_with_candidate
                self.ensemble_model.promote_candidate()
                self.best_model_size = len(self.ensemble_model.models)
                self.best_ensemble_state = copy.deepcopy(self.ensemble_model.state_dict())
                logging.info(f"候选模型被采纳! 当前最佳RMSE: {self.best_test_rmse:.6f}, 集成模型数量: {self.best_model_size}")
            else:
                self.ensemble_model.discard_candidate()
                logging.info(f"候选模型被拒绝。当前最佳RMSE: {self.best_test_rmse:.6f}")

        return self.best_test_rmse, self.best_model_size

    def evaluate(self, dataloader):
        """在给定数据集上评估模型并返回 RMSE"""
        self.ensemble_model.eval()
        total_mse = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in dataloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.ensemble_model(inputs)
                total_mse += criterion(outputs, targets).item() * inputs.size(0)
        return np.sqrt(total_mse / len(dataloader.dataset))

# CFNet-Boost 网格搜索函数
def cfnet_boost_grid_search(depths, degrees, base_hparams, results_path='cfnet_boost_grid_results.json'):
    """运行 CFNet-Boost 在不同深度和阶数组合下的网格搜索"""
    setup_logging('cfnet_boost_grid_search.log')
    
    # 数据准备
    (X_train, y_train), (X_test, y_test), input_dim = get_regression_data_loaders(batch_size=base_hparams['batch_size'])
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    
    base_hparams['input_dim'] = input_dim
    
    # 存储结果
    results_rmse = np.zeros((len(degrees), len(depths)), dtype=float)
    results_num_models = np.zeros((len(degrees), len(depths)), dtype=int)
    training_times = np.zeros((len(degrees), len(depths)), dtype=float)
    details = {}
    
    total_combinations = len(depths) * len(degrees)
    current_combination = 0
    
    logging.info("开始 CFNet-Boost 网格搜索实验...")
    logging.info(f"深度范围: {depths}")
    logging.info(f"阶数范围: {degrees}")
    logging.info(f"总组合数: {total_combinations}")
    
    for i, degree in enumerate(degrees):
        for j, depth in enumerate(depths):
            current_combination += 1
            logging.info(f"进度: {current_combination}/{total_combinations} - 测试阶数={degree}, 深度={depth}")
            
            # 更新超参数
            hparams = base_hparams.copy()
            hparams['polynomial_degree'] = degree
            hparams['shallow_depth_per_cofrnet'] = depth
            
            # 创建模型
            regressor = CoFrNetRegressor(hparams)
            
            # 训练模型
            start_time = time.time()
            best_rmse, best_model_size = regressor.train(
                train_data=(X_train, y_train),
                val_data=(X_val, y_val),
                test_data=(X_test, y_test),
                max_attempts=hparams['num_models_max']
            )
            training_time = time.time() - start_time
            
            # 存储结果
            results_rmse[i, j] = best_rmse
            results_num_models[i, j] = best_model_size
            training_times[i, j] = training_time
            
            details[f"{degree}_{depth}"] = {
                "rmse": float(best_rmse),
                "ensemble_size": int(best_model_size),
                "training_time": float(training_time)
            }
            
            logging.info(f"阶数={degree}, 深度={depth}: RMSE={best_rmse:.6f}, 模型数量={best_model_size}, 训练时间={training_time:.2f}s")
            
            # 清理内存
            del regressor
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # 保存结果到JSON文件
    results_dict = {
        'depths': depths,
        'degrees': degrees,
        'results_rmse': results_rmse.tolist(),
        'results_num_models': results_num_models.tolist(),
        'training_times': training_times.tolist(),
        'details': details
    }
    
    with open(results_path, 'w') as f:
        json.dump(results_dict, f, indent=2)
    
    logging.info(f"网格搜索结果已保存到 {results_path}")
    
    return results_rmse, results_num_models, training_times, degrees, depths

# 绘制热力图
def plot_cfnet_boost_heatmaps(results_rmse, results_num_models, training_times, degrees, depths,save_path='OnlineNewsPopularity_heatmap_depth_vs_degree.png'):
    """绘制 CFNet-Boost 网格搜索结果热力图"""

    # 绘制热力图
    plt.figure(figsize=(10, 8))
    # RMSE 热力图
    sns.heatmap(results_rmse, annot=True, fmt=".4f", cmap='viridis_r', 
                xticklabels=depths, yticklabels=degrees, ax=axes[0, 0],cbar_kws={'label': 'RMSE'})
    plt.xlabel('Depth (shallow depth per CFNet)', fontsize=12)
    plt.ylabel('Polynomial Degree', fontsize=12)
    plt.title('OnlineNewsPopularity CFNet-Boost Grid Search RMSE\n(lower is better)', 
              fontsize=14, pad=20)  
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight') 
    logging.info(f'CFNet-Boost热力图已保存 -> {save_path}')

# 分析最佳参数
def analyze_best_cfnet_boost_parameters(results_rmse, degrees, depths):
    """分析 CFNet-Boost 最佳参数组合"""
    min_rmse_idx = np.unravel_index(np.argmin(results_rmse), results_rmse.shape)
    best_degree = degrees[min_rmse_idx[0]]
    best_depth = depths[min_rmse_idx[1]]
    best_rmse = results_rmse[min_rmse_idx]
    
    logging.info("=" * 50)
    logging.info("CFNet-Boost 最佳参数分析:")
    logging.info(f"最佳多项式阶数: {best_degree}")
    logging.info(f"最佳深度: {best_depth}")
    logging.info(f"最佳RMSE: {best_rmse:.6f}")
    logging.info("=" * 50)
    
    # # 绘制3D曲面图
    # fig = plt.figure(figsize=(12, 10))
    # ax = fig.add_subplot(111, projection='3d')
    
    # X, Y = np.meshgrid(depths, degrees)
    # surf = ax.plot_surface(X, Y, results_rmse, cmap='viridis', alpha=0.8)
    
    # # 标记最佳点
    # ax.scatter(best_depth, best_degree, best_rmse, color='red', s=100, label='Best Point')
    
    # ax.set_xlabel('Depth (shallow depth per CFNet)')
    # ax.set_ylabel('Polynomial Degree')
    # ax.set_zlabel('RMSE')
    # ax.set_title('3D Surface Plot of CFNet-Boost RMSE vs Degree and Depth')
    # ax.legend()
    
    # plt.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    # plt.savefig('cfnet_boost_3d_surface_plot.png', dpi=300, bbox_inches='tight')
    # plt.show()
    
    return best_degree, best_depth, best_rmse

# 主函数
if __name__ == '__main__':
    # 基础超参数配置
    BASE_HPARAMS = {
        "task_name": "OnlineNewsPopularity",
        "num_models_max": 10,  # 每个配置的最大模型数量
        "loss_target": 0, 
        "epochs_per_model": 200,  # 每个模型的训练轮数
        "boosting_learning_rate": 0.1,
        "learning_rate_adam": 1e-3,
        "weight_decay": 1e-5,
        "batch_size": 128,
        "early_stopping_patience": 10,
        "input_dim": None,  # 将在运行时设置
        "output_dim": 1
    }
    
    # 定义搜索范围
    depths = [1, 2, 3, 4, 5, 6, 7, 8]
    degrees = [1, 2, 3, 4, 5, 6, 7, 8]
    
    # 运行网格搜索
    results_rmse, results_num_models, training_times, degrees, depths = cfnet_boost_grid_search(
        depths, degrees, BASE_HPARAMS
    )
    
    # 绘制热力图
    plot_cfnet_boost_heatmaps(results_rmse, results_num_models, training_times, degrees, depths,save_path='OnlineNewsPopularity_heatmap_depth_vs_degree.png')
    
    # 分析最佳参数
    best_degree, best_depth, best_rmse = analyze_best_cfnet_boost_parameters(results_rmse, degrees, depths)
    
    logging.info("CFNet-Boost 网格搜索实验完成！")